In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [ ]:
counts = train["answer"].value_counts()

print(counts)

most_frequent = counts.max()
least_frequent = counts.min()

result = most_frequent + least_frequent

print("Answer:", result)

In [ ]:
counts = train["answer"].value_counts()

print(counts)

most_frequent = counts.max()
least_frequent = counts.min()

result = most_frequent + least_frequent

print("Answer:", result)

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Get prompt for Row ID 1
prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

# Clean
prompt = prompt.lower()
prompt = prompt.translate(str.maketrans("", "", string.punctuation))

# Tokenize
words = prompt.split()

# Remove stop words
filtered_words = [
    word for word in words
    if word not in ENGLISH_STOP_WORDS
]

print("Remaining words:", len(filtered_words))
print(filtered_words)

In [ ]:
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(combined_text)

print("Vocabulary size:", X.shape[1])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

option_a = str(row["A"])

# Fit TF-IDF on just the prompt and option A
vectorizer = TfidfVectorizer()

tfidf = vectorizer.fit_transform([prompt, option_a])

similarity = cosine_similarity(
    tfidf[0:1],
    tfidf[1:2]
)[0][0]

print(round(similarity, 4))

In [ ]:
# Build corpus
from sklearn.feature_extraction.text import TfidfVectorizer
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

# Fit TF-IDF
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

correct_matches = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([str(row["prompt"])])

    similarities = {}

    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform([str(row[option])])

        similarities[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    predicted = max(similarities, key=similarities.get)

    if predicted == row["answer"]:
        correct_matches += 1

percentage = (correct_matches / len(train)) * 100

print("Correct Matches:", correct_matches)
print("Percentage:", round(percentage, 2))

In [ ]:
# Build corpus
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

# Fit TF-IDF
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

def ap3(actual, preds):
    if actual in preds[:3]:
        return 1.0 / (preds[:3].index(actual) + 1)
    return 0.0

scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([str(row["prompt"])])

    sims = {}

    for opt in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform([str(row[opt])])

        sims[opt] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked = sorted(
        sims,
        key=sims.get,
        reverse=True
    )

    scores.append(
        ap3(row["answer"], ranked)
    )

map3_score = sum(scores) / len(scores)

print(round(map3_score, 6))